# Проверка практической полезности вероятностных моделей

Ноутбук загружает последний или указанный запуск из отдельного S3-раздела, сравнивает стратегии после комиссии и показывает устойчивость результата к порогу вероятности.

In [ ]:
from __future__ import annotations
import io, json, sys
from pathlib import Path
import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'build_price_feature_day.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from build_price_feature_day import make_s3_client

BUCKET = 'binance-data-downloader'
ROOT_PREFIX = 'trading_strategy_experiment'
RUN_ID = None  # Укажите конкретный run_id или оставьте None для latest.json
s3 = make_s3_client()

In [ ]:
def read_json(key):
    return json.loads(s3.get_object(Bucket=BUCKET, Key=key)['Body'].read().decode('utf-8'))

def read_parquet(key):
    payload = s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()
    return pd.read_parquet(io.BytesIO(payload))

if RUN_ID is None:
    RUN_ID = read_json(f'{ROOT_PREFIX}/latest.json')['run_id']
RUN_PREFIX = f'{ROOT_PREFIX}/{RUN_ID}'
config = read_json(f'{RUN_PREFIX}/run_config.json')
metrics = read_parquet(f'{RUN_PREFIX}/all_metrics.parquet')
selected = read_parquet(f'{RUN_PREFIX}/selected_evaluation_results.parquet')
display(config)
display(selected.sort_values(['horizon', 'strategy']))

## Итог на независимой evaluation-части

In [ ]:
evaluation = metrics.query("segment == 'evaluation'").copy()
point = evaluation[evaluation.strategy.isin(['point_sign', 'point_cost_threshold'])]
comparison = pd.concat([point, selected], ignore_index=True, sort=False)
display(comparison[['horizon','strategy','trades','net_total_return','max_drawdown','win_rate','profit_factor']].sort_values(['horizon','strategy']))

pivot = comparison.pivot(index='horizon', columns='strategy', values='net_total_return')
ax = pivot.plot.bar(figsize=(13, 5), title='Чистая доходность после комиссии')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Net total return')
plt.tight_layout()

## Чувствительность к порогу вероятности

In [ ]:
prob = metrics[metrics.strategy.str.contains('_p', regex=False)].copy()
prob['model'] = np.where(prob.strategy.str.startswith('stat_garch'), 'Stat + GARCH', 'CatBoost MQ')
prob['threshold'] = prob.strategy.str.rsplit('_p', n=1).str[-1].astype(float)
fig, axes = plt.subplots(len(config['horizons']), 2, figsize=(13, 4 * len(config['horizons'])), squeeze=False)
for row, horizon in enumerate(config['horizons']):
    part = prob[(prob.horizon == horizon) & (prob.segment == 'evaluation')]
    for model, group in part.groupby('model'):
        group = group.sort_values('threshold')
        axes[row, 0].plot(group.threshold, group.net_total_return, marker='o', label=model)
        axes[row, 1].plot(group.threshold, group.trades, marker='o', label=model)
    axes[row, 0].axhline(0, color='black', linewidth=.8)
    axes[row, 0].set_title(f'{horizon} мин: доходность')
    axes[row, 1].set_title(f'{horizon} мин: число сделок')
    axes[row, 0].legend(); axes[row, 1].legend()
plt.tight_layout()

## Кривые капитала выбранных стратегий

In [ ]:
fig, axes = plt.subplots(len(config['horizons']), 1, figsize=(14, 4 * len(config['horizons'])), squeeze=False)
for row, horizon in enumerate(config['horizons']):
    equity = read_parquet(f'{RUN_PREFIX}/horizon_{horizon}/equity.parquet')
    chosen = set(selected.loc[selected.horizon.eq(horizon), 'strategy']) | {'point_sign', 'point_cost_threshold'}
    part = equity[(equity.segment == 'evaluation') & equity.strategy.isin(chosen)].copy()
    for strategy, group in part.groupby('strategy'):
        axes[row, 0].plot(pd.to_datetime(group.exit_timestamp), group.equity, label=strategy)
    axes[row, 0].set_title(f'{horizon} минут')
    axes[row, 0].legend()
plt.tight_layout()

## Интерпретация

Основной вывод следует делать по `evaluation`, а не по `validation`. Положительный результат должен сопровождаться достаточным числом сделок, приемлемой просадкой и устойчивостью к соседним порогам. Тест использует целевую доходность как прокси исполнения и не учитывает проскальзывание и funding.